### crear src/servidor.py: librerías, carpetas, red completa y nombres de las clases

In [1]:
%%writefile ../src/servidor.py
from pathlib import Path
import base64
import cv2
import numpy as np
import pandas as pd
import torch
from fastapi import FastAPI
from fastapi.responses import RedirectResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

from red import cargar_red, activaciones
from procesamiento import limpiar_foto, sacar_letras

Carpeta_Proyecto = Path(__file__).parent.parent
Carpeta_modelos = Carpeta_Proyecto / "modelos"
Carpeta_web = Carpeta_Proyecto / "web"

tabla_clases = pd.read_csv(Carpeta_modelos / "clases_red_completa.csv", dtype=str)
clases = tabla_clases["clase"].tolist()
grupos = tabla_clases["grupo"].tolist()
es_animal = np.array([g == "animal" for g in grupos])
red = cargar_red(Carpeta_modelos / "red_completa.pt", n_salidas=len(clases))

UMBRAL = 0.5
NO_RECONOCIDO = "No se pudo reconocer el dibujo"

app = FastAPI()


class Pedido(BaseModel):
    imagen: str
    origen: str = "pizarra"


@app.get("/estado")
def estado():
    return {"mensaje": "servidor encendido", "clases": len(clases)}

Overwriting ../src/servidor.py


### agregar la función que convierte la imagen que manda el celular en una imagen de OpenCV

In [2]:
%%writefile -a ../src/servidor.py


def leer_imagen(texto_imagen):
    datos = base64.b64decode(texto_imagen.split(",")[-1])
    return cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)

Appending to ../src/servidor.py


### agregar la función que convierte un dibujo completo en una imagen de 28×28

In [3]:
%%writefile -a ../src/servidor.py


def dibujo_a_28(sin_bordes):
    filas, columnas = np.where(sin_bordes > 0)
    if len(filas) == 0:
        return None
    recorte = sin_bordes[filas.min():filas.max() + 1, columnas.min():columnas.max() + 1]
    alto_r, ancho_r = recorte.shape
    escala = 20 / max(alto_r, ancho_r)
    nuevo_ancho_r = max(1, int(ancho_r * escala))
    nuevo_alto_r = max(1, int(alto_r * escala))
    chico = cv2.resize(recorte, (nuevo_ancho_r, nuevo_alto_r), interpolation=cv2.INTER_AREA)
    imagen_28 = np.zeros((28, 28), np.uint8)
    arriba = (28 - nuevo_alto_r) // 2
    izquierda = (28 - nuevo_ancho_r) // 2
    imagen_28[arriba:arriba + nuevo_alto_r, izquierda:izquierda + nuevo_ancho_r] = chico
    return imagen_28

Appending to ../src/servidor.py


### agregar la función que prepara las capas de la red para la animación del celular

In [4]:
%%writefile -a ../src/servidor.py

NOMBRES_CAPAS = {
    "conv1": "Convolución 1",
    "conv2": "Convolución 2",
    "densa1": "Densa 1",
    "densa2": "Densa 2",
    "salida": "Salida",
}


def capas_para_panel(capas):
    lista = []
    for clave, nombre in NOMBRES_CAPAS.items():
        valores = capas[clave]
        maximo = valores.max()
        if maximo > 0:
            valores = valores / maximo
        lista.append({"nombre": nombre, "valores": valores.round(3).tolist()})
    return lista

Appending to ../src/servidor.py


### agregar las funciones que arman la frase de la respuesta y las otras opciones

In [5]:
%%writefile -a ../src/servidor.py


def mensaje_para(clase, grupo):
    if grupo == "dígito":
        return f"Es el número {clase}"
    if grupo == "letra":
        return f"Es la letra {clase}"
    return f"Es un {clase}"


def opciones_de(probabilidades, cantidad=3):
    mejores = probabilidades.argsort()[::-1][:cantidad]
    return [{"clase": clases[k], "probabilidad": round(float(probabilidades[k]), 4)} for k in mejores]

Appending to ../src/servidor.py


### agregar la respuesta para cuando la red no reconoce lo que se envió

In [6]:
%%writefile -a ../src/servidor.py


def respuesta_no_reconocida(capas=None, confianza=0.0):
    return {
        "reconocido": False,
        "respuesta": "?",
        "mensaje": NO_RECONOCIDO,
        "categoria": NO_RECONOCIDO,
        "confianza": round(confianza, 4),
        "opciones": [],
        "capas": capas_para_panel(capas) if capas else [],
    }

Appending to ../src/servidor.py


### agregar la función que reconoce un dibujo de la pizarra: número, letra o animal

In [7]:
%%writefile -a ../src/servidor.py


def leer_dibujo(sin_bordes):
    imagen_28 = dibujo_a_28(sin_bordes)
    if imagen_28 is None:
        return respuesta_no_reconocida()

    capas = activaciones(red, imagen_28)
    probabilidades = capas["salida"]
    k = int(probabilidades.argmax())
    confianza = float(probabilidades[k])
    if confianza < UMBRAL:
        return respuesta_no_reconocida(capas, confianza)

    return {
        "reconocido": True,
        "respuesta": clases[k],
        "mensaje": mensaje_para(clases[k], grupos[k]),
        "categoria": grupos[k],
        "confianza": round(confianza, 4),
        "opciones": opciones_de(probabilidades),
        "capas": capas_para_panel(capas),
    }

Appending to ../src/servidor.py


### agregar la función que reconoce cada carácter de una foto de hoja, sin animales

In [8]:
%%writefile -a ../src/servidor.py


def reconocer_caracteres(imagenes):
    x = torch.tensor(np.array(imagenes) / 255.0, dtype=torch.float32).reshape(-1, 1, 28, 28)
    with torch.no_grad():
        probabilidades = torch.softmax(red(x), dim=1).numpy()
    probabilidades[:, es_animal] = 0
    return probabilidades

Appending to ../src/servidor.py


### agregar la función que arma el texto de la hoja con sus renglones y palabras

In [9]:
%%writefile -a ../src/servidor.py


def armar_texto(texto_letras, elegidas):
    posicion = 0
    lineas = []
    for linea in texto_letras:
        palabras = []
        for palabra in linea:
            caracteres = ""
            for _ in palabra:
                caracteres += clases[elegidas[posicion]]
                posicion += 1
            palabras.append(caracteres)
        lineas.append(" ".join(palabras))
    return "\n".join(lineas)

Appending to ../src/servidor.py


### agregar la función que lee una foto de hoja completa

In [10]:
%%writefile -a ../src/servidor.py


def leer_hoja(sin_bordes):
    try:
        texto_letras = sacar_letras(sin_bordes)
    except Exception:
        return respuesta_no_reconocida()

    imagenes = [letra for linea in texto_letras for palabra in linea for letra in palabra]
    if not imagenes:
        return respuesta_no_reconocida()

    probabilidades = reconocer_caracteres(imagenes)
    elegidas = probabilidades.argmax(1)
    texto = armar_texto(texto_letras, elegidas)
    confianza = float(probabilidades.max(1).mean())
    capas = activaciones(red, imagenes[0])

    return {
        "reconocido": True,
        "respuesta": "Texto leído",
        "mensaje": texto,
        "categoria": "hoja",
        "confianza": round(confianza, 4),
        "opciones": [],
        "texto": texto,
        "capas": capas_para_panel(capas),
    }

Appending to ../src/servidor.py


### agregar el camino /predecir, que recibe el dibujo o la foto y responde

In [11]:
%%writefile -a ../src/servidor.py


@app.post("/predecir")
def predecir_imagen(pedido: Pedido):
    foto = leer_imagen(pedido.imagen)
    if foto is None:
        return respuesta_no_reconocida()
    sin_bordes = limpiar_foto(foto)
    if pedido.origen == "foto":
        return leer_hoja(sin_bordes)
    return leer_dibujo(sin_bordes)

Appending to ../src/servidor.py


### agregar la entrega de las páginas web: al abrir el servidor se va directo a la página del celular

In [12]:
%%writefile -a ../src/servidor.py


@app.get("/")
def inicio():
    return RedirectResponse("/celular/")


app.mount("/", StaticFiles(directory=Carpeta_web, html=True), name="web")

Appending to ../src/servidor.py
